### Import Libraries

In [1]:
import re
import time
from collections import Counter

import pandas as pd
import requests

### Functions

In [2]:
URL = "https://titanic-agent-401145028214.us-central1.run.app"

In [3]:
df = pd.read_csv("https://raw.githubusercontent.com/mwaskom/seaborn-data/master/titanic.csv")

In [4]:
def ask_agent(question):
    """Return (output, error): exactly one of the two is None."""
    try:
        r = requests.post(URL, json={"question": question}, timeout=300)
    except requests.RequestException as e:
        # timeout, connection reset, DNS... (no HTTP response at all)
        return None, f"{type(e).__name__}: {e}"

    if not r.ok:
        # the API answers {"error": "..."}; Cloud Run itself (403, 429, 503, 504) answers HTML/plain text
        try:
            detail = r.json().get("error", r.text)
        except ValueError:
            detail = r.text.strip()[:500]
        return None, f"HTTP {r.status_code} {r.reason}: {detail}"

    return r.json(), None

In [5]:
def numbers_in(text):
    """All numbers that appear in a text (handles 1,234 and 12.5%)."""
    return [float(n.replace(",", "")) for n in re.findall(r"\d[\d,]*\.?\d*", text)]

In [6]:
def is_correct(answer, expected, rel_tol=0.01):
    """True if any number in the answer is within rel_tol of the expected value."""
    return any(abs(n - expected) <= max(rel_tol * abs(expected), 0.05) for n in numbers_in(answer))

### Single question

In [7]:
output, error = ask_agent("How many passengers survived?")

if error:
    print(error)
else:
    print("Answer:", output["answer"])
    print("\nTools used:")
    for tc in output["tool_calls"]:
        print(f"  - {tc['name']}({tc['args']})")

Answer: A total of 342 passengers survived.

Tools used:
  - schema({})
  - filter_passengers({'condition': 'survived == 1'})


### Test set (questions + ground truth computed with pandas)

In [8]:
test_set = [
    ("How many passengers are in the dataset?",                              len(df)),
    ("How many passengers survived?",                                        df["survived"].sum()),
    ("What was the overall survival rate, as a percentage?",                 df["survived"].mean() * 100),
    ("What was the survival rate of women, as a percentage?",                df.loc[df["sex"] == "female", "survived"].mean() * 100),
    ("What was the survival rate of men, as a percentage?",                  df.loc[df["sex"] == "male", "survived"].mean() * 100),
    ("What was the survival rate of first class passengers, as a percentage?", df.loc[df["pclass"] == 1, "survived"].mean() * 100),
    ("What was the survival rate of third class passengers, as a percentage?", df.loc[df["pclass"] == 3, "survived"].mean() * 100),
    ("What was the average age of the passengers?",                          df["age"].mean()),
    ("How many passengers were under 18 years old?",                         (df["age"] < 18).sum()),
    ("How many passengers embarked at Southampton?",                         (df["embark_town"] == "Southampton").sum()),
    ("What was the average fare paid?",                                      df["fare"].mean()),
]

for q, expected in test_set:
    print(f"{expected:>8.2f}  {q}")

  891.00  How many passengers are in the dataset?
  342.00  How many passengers survived?
   38.38  What was the overall survival rate, as a percentage?
   74.20  What was the survival rate of women, as a percentage?
   18.89  What was the survival rate of men, as a percentage?
   62.96  What was the survival rate of first class passengers, as a percentage?
   24.24  What was the survival rate of third class passengers, as a percentage?
   29.70  What was the average age of the passengers?
  113.00  How many passengers were under 18 years old?
  644.00  How many passengers embarked at Southampton?
   32.20  What was the average fare paid?


### Inference

In [9]:
N = 20
c = 1

results = []
failed = []
start = time.perf_counter()
t_block = start

for i, (question, expected) in enumerate(test_set[:N], start=1):
    t0 = time.perf_counter()
    output, error = ask_agent(question)
    elapsed = time.perf_counter() - t0

    if error is not None:
        failed.append((i - 1, error))
        print(f"Iteration {i} FAILED -> {error}")
    else:
        results.append({
            "question": question,
            "expected": round(float(expected), 2),
            "answer": output["answer"],
            "tools": [tc["name"] for tc in output["tool_calls"]],
            "n_tool_calls": len(output["tool_calls"]),
            "correct": is_correct(output["answer"], float(expected)),
            "seconds": round(elapsed, 2),
        })

    if i % c == 0:
        now = time.perf_counter()
        print(f"Iterations {i - c + 1}-{i}: {now - t_block:.2f} s "
              f"(elapsed: {now - start:.2f} s)")
        t_block = now

print(f"Total: {N} iterations in {time.perf_counter() - start:.2f} s "
      f"({len(failed)} failed)")

for idx, error in failed:
    print(f"[{idx}] {error}")

res = pd.DataFrame(results)

Iterations 1-1: 2.02 s (elapsed: 2.02 s)
Iterations 2-2: 3.65 s (elapsed: 5.66 s)
Iterations 3-3: 5.68 s (elapsed: 11.34 s)
Iterations 4-4: 5.81 s (elapsed: 17.16 s)
Iterations 5-5: 8.46 s (elapsed: 25.62 s)
Iterations 6-6: 4.88 s (elapsed: 30.50 s)
Iterations 7-7: 3.92 s (elapsed: 34.42 s)
Iterations 8-8: 3.04 s (elapsed: 37.46 s)
Iterations 9-9: 2.55 s (elapsed: 40.01 s)
Iterations 10-10: 2.78 s (elapsed: 42.79 s)
Iterations 11-11: 2.61 s (elapsed: 45.40 s)
Total: 20 iterations in 45.40 s (0 failed)


### Performance

In [10]:
print(f"Accuracy:            {res['correct'].mean():.4f} ({res['correct'].sum()}/{len(res)})")
print(f"Avg latency:         {res['seconds'].mean():.2f} s")
print(f"Avg tool calls:      {res['n_tool_calls'].mean():.2f}")

pd.set_option("display.max_colwidth", 120)
res[["question", "expected", "answer", "correct", "seconds"]]

Accuracy:            1.0000 (11/11)
Avg latency:         4.13 s
Avg tool calls:      3.09


,question,expected,answer,correct,seconds
0,How many passengers are in the dataset?,891.00,There are 891 passengers in the dataset.,True,2.02
1,How many passengers survived?,342.00,A total of 342 passengers survived.,True,3.65
2,"What was the overall survival rate, as a percentage?",38.38,The overall survival rate of the Titanic passengers was approximately **38.38%** (or 38.4%).,True,5.68
3,"What was the survival rate of women, as a percentage?",74.20,The survival rate for women on the Titanic depends on how they are defined:\n\n* **All female passengers (including ...,True,5.81
4,"What was the survival rate of men, as a percentage?",18.89,The survival rate of men on the Titanic depends on how they are defined:\n\n* **All male passengers (including child...,True,8.46
5,"What was the survival rate of first class passengers, as a percentage?",62.96,"The survival rate of first class passengers was **62.96%** (or approximately **63.0%**), with 136 out of 216 passeng...",True,4.88
6,"What was the survival rate of third class passengers, as a percentage?",24.24,The survival rate of third-class passengers was **24.2%** (119 out of 491 passengers survived).,True,3.92
7,What was the average age of the passengers?,29.70,The average age of the passengers was approximately 29.7 years (29.70).,True,3.04
8,How many passengers were under 18 years old?,113.00,There were 113 passengers under 18 years old.,True,2.55
9,How many passengers embarked at Southampton?,644.00,644 passengers embarked at Southampton.,True,2.78
